# AAM ground-truth preprocessor
**Download → clean → export one CSV per song → train the HMM**

Run **steps 1–6** to download and clean annotations for songs **0001–1000**, then export them to Google Drive. Run `AAM_BeatNet_HMM_Colab.ipynb` afterward: it reads the same `aam_cleaned_csv` folder.

The exported `chord` and `key` remain in their **original pitch**, matching the audio. C-normalized versions are included separately as `chord_C` and `key_C` for inspection. The HMM uses the original chord and performs its own chord-root rotation internally.

All tempos and songs are retained. Original ARFF files are never overwritten. **Steps 7–8 are optional** inspection and debugging after export.


## 1. Set folders and download annotations
This Colab notebook downloads the [first-1,000 AAM annotations](https://zenodo.org/records/5794629) directly from Zenodo. It does not require an existing `full_data` dataset folder. Only annotation files are needed here; the HMM notebook downloads audio when it needs it.

The downloaded ZIP and extracted source files are temporary, under `/content/aam`. Cleaned CSVs are saved to Drive. **Use the same `PROJECT` / `project_path` in both notebooks.** Both use `aam_cleaned_csv` for their connection.


In [ ]:
from pathlib import Path
import csv
import subprocess
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Femman/DT2470 Music Informatics/DT2470 Project/full_data")
OUTPUT = PROJECT / "aam_cleaned_csv"
ANNOTATIONS = Path("/content/aam/annotations")
ANNOTATIONS.mkdir(parents=True, exist_ok=True)

annotation_url = "https://zenodo.org/records/5794629/files/0001-1000-annotations-v1.1.0.zip?download=1"
annotation_zip = Path("/content/aam/annotations.zip")

if not zipfile.is_zipfile(annotation_zip):
    temporary = annotation_zip.with_suffix(".part")
    subprocess.run(["wget", "--tries=3", annotation_url, "-O", str(temporary)], check=True)
    assert zipfile.is_zipfile(temporary), "Invalid ZIP: rerun the download cell."
    temporary.replace(annotation_zip)

# Read only the two annotation types used for cleaning.
with zipfile.ZipFile(annotation_zip) as archive:
    for name in archive.namelist():
        if name.endswith(("_beatinfo.arff", "_segments.arff")):
            file_path = ANNOTATIONS / Path(name).name
            file_path.write_bytes(archive.read(name))

# Timestamps have small rounding differences (0.1 ms).
TIME_TOL = 0.0001
print("Source annotations:", ANNOTATIONS)
print("Cleaned output for the HMM:", OUTPUT)


Mounted at /content/drive
Source annotations: /content/aam/annotations
Cleaned output for the HMM: /content/drive/MyDrive/Femman/DT2470 Music Informatics/DT2470 Project/full_data/aam_cleaned_csv


## 2. Read the annotations
The ARFF rows are comma-separated, with single quotes around text. Ignore header lines and give the four columns we need short names. Some `beatinfo` files omit `@DATA`; this reader handles them too.

In [ ]:
def read_arff(file_path, columns):
    rows = []

    with open(file_path, encoding="utf-8-sig") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith(("@", "%")):
                continue
            row = next(csv.reader([line], quotechar="'", skipinitialspace=True))
            rows.append(row[:len(columns)])

    return pd.DataFrame(rows, columns=columns)


def find_files(suffix):
    files = {}
    for file_path in sorted(ANNOTATIONS.rglob(f"*_{suffix}.arff")):
        song_id = file_path.name.split("_")[0]
        assert song_id not in files, f"Duplicate {suffix} file: {song_id}"
        files[song_id] = file_path
    return files


beat_files = find_files("beatinfo")
segment_files = find_files("segments")
expected_ids = {f"{i:04d}" for i in range(1, 1001)}
assert set(beat_files) == set(segment_files) == expected_ids, "Expected songs 0001–1000"

raw_beats = {}
segments = {}

for song_id in sorted(beat_files):
    beats = read_arff(beat_files[song_id], ["time_s", "bar", "quarter", "chord"])
    for column in ["time_s", "bar", "quarter"]:
        beats[column] = pd.to_numeric(beats[column])

    parts = read_arff(segment_files[song_id], ["time_s", "section", "bpm", "key"])
    for column in ["time_s", "bpm"]:
        parts[column] = pd.to_numeric(parts[column])

    raw_beats[song_id] = beats
    segments[song_id] = parts

print("Songs loaded:", len(raw_beats))

Songs loaded: 1000


## 3. Remove extra bars
Process segments in their original row order. Each segment begins at `bar = 1, quarter = 1`. Its expected number of quarters is **duration × BPM / 60**.

Keep that many rows. Before removing a tail, check that it is exactly one extra bar on the expected beat grid. Unexpected data stops processing before export. The calculation and checks are separated from the loop to keep both readable.

In [ ]:
def check_segment(block, segment, end_time):
    exact_count = (end_time - segment.time_s) * segment.bpm / 60
    count = int(round(exact_count))
    assert count > 0 and count % 4 == 0, "Expected whole 4/4 bars"
    assert abs(exact_count - count) <= 0.001, "Segment duration does not match BPM"
    assert len(block) - count in (0, 4), "Expected zero or one surplus bar"

    positions = np.arange(len(block))
    expected_times = segment.time_s + positions * 60 / segment.bpm
    assert np.allclose(block.time_s, expected_times, atol=TIME_TOL, rtol=0), "Beat times off grid"
    assert np.array_equal(block.bar, positions // 4 + 1), "Unexpected bar numbers"
    assert np.array_equal(block.quarter, positions % 4 + 1), "Unexpected quarter numbers"
    if len(block) > count:
        assert block.time_s.iloc[count] >= end_time - TIME_TOL, "Surplus starts inside segment"
    return count


def remove_extra_bars(beats, parts):
    assert len(parts) >= 2 and parts.section.iloc[-1].lower() == "end"
    assert parts.section.str.lower().eq("end").sum() == 1
    assert np.isfinite(parts[["time_s", "bpm"]]).all().all()
    assert abs(parts.time_s.iloc[0]) <= TIME_TOL
    assert (np.diff(parts.time_s) > 0).all() and (parts.bpm.iloc[:-1] > 0).all()
    assert np.isfinite(beats[["time_s", "bar", "quarter"]]).all().all()
    assert beats.chord.notna().all() and beats.chord.str.strip().ne("").all()

    starts = np.flatnonzero((beats.bar == 1) & (beats.quarter == 1))
    assert len(starts) == len(parts) - 1 and starts[0] == 0
    stops = np.append(starts[1:], len(beats))
    kept_blocks = []
    removed_blocks = []

    for i, (start, stop) in enumerate(zip(starts, stops)):
        block = beats.iloc[start:stop].copy()
        count = check_segment(block, parts.iloc[i], parts.time_s.iloc[i + 1])
        block["segment"] = i + 1
        kept_blocks.append(block.iloc[:count])
        removed_blocks.append(block.iloc[count:])

    return pd.concat(kept_blocks, ignore_index=True), pd.concat(removed_blocks, ignore_index=True)

In [ ]:
cleaned_beats = {}
removed_rows = {}

for song_id in sorted(raw_beats):
    try:
        clean, removed = remove_extra_bars(raw_beats[song_id], segments[song_id])
    except AssertionError as error:
        raise ValueError(f"Song {song_id}: {error}") from error
    cleaned_beats[song_id] = clean
    removed_rows[song_id] = removed

print("Extra-bar cleaning finished.")

Extra-bar cleaning finished.


## 4. Add interval and segment fields
Attach each segment's section, BPM and original key to its retained beats. A quarter ends at the next retained beat; the final quarter ends at the song's `end` marker. Bar numbers still restart within each segment.

These start/end times are exported directly, so the HMM notebook does not need to download segment annotations again.


In [ ]:
cleaned_songs = {}

for song_id, beats in cleaned_beats.items():
    song = beats.copy()
    parts = segments[song_id].iloc[:-1]

    for column in ["section", "bpm", "key"]:
        # Segment 1 refers to row 0 in the segment table.
        values = dict(zip(range(1, len(parts) + 1), parts[column]))
        song[column] = song.segment.map(values)

    song["end_time_s"] = np.append(song.time_s.to_numpy()[1:], segments[song_id].time_s.iloc[-1])
    song[["bar", "quarter"]] = song[["bar", "quarter"]].astype(int)
    assert (np.diff(song.time_s) > 0).all(), song_id
    assert (song.end_time_s > song.time_s).all(), song_id

    cleaned_songs[song_id] = song[[
        "time_s", "end_time_s", "segment", "section", "bar", "quarter", "chord", "bpm", "key"
    ]]

print("Interval and segment fields added.")

Interval and segment fields added.


## 5. Add C-normalized labels for inspection
Calculate each segment's downward semitone shift, preserving chord quality and `N.C.`. For example, D major shifts by −2 to C major.

Keep this transformation separate from `cleaned_songs`, which has the original-pitch labels. At export, the normalized labels become **`chord_C` and `key_C`**. The connected HMM uses **`chord`**, matching original audio; it does not use the ground-truth key to normalize a test song.


In [ ]:
pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def transpose_label(label, shift):
    if label == "N.C.":
        return label
    root, quality = label[:-3], label[-3:]
    assert root in pitch_classes and quality in ["maj", "min"], f"Unknown label: {label}"
    new_root = pitch_classes[(pitch_classes.index(root) + shift) % 12]
    return new_root + quality


transposed_songs = {}

for song_id, clean in cleaned_songs.items():
    song = clean.copy()
    song["original_key"] = song.key
    song["original_chord"] = song.chord

    key_shifts = {}
    for key in song.key.unique():
        transpose_label(key, 0)  # check that the key is supported
        key_shifts[key] = -pitch_classes.index(key[:-3])

    song["transpose_semitones"] = song.key.map(key_shifts)
    for column in ["key", "chord"]:
        song[column] = [transpose_label(label, shift)
                        for label, shift in zip(song[column], song.transpose_semitones)]

    assert song.key.isin(["Cmaj", "Cmin"]).all(), song_id
    transposed_songs[song_id] = song

print("All segment keys normalized to C.")

All segment keys normalized to C.


## 6. Export one cleaned CSV per song
Files are named `0001_ground_truth.csv`, …, `1000_ground_truth.csv` in **`PROJECT / "aam_cleaned_csv"`**. The HMM notebook reads that folder directly. Run this cell before opening the HMM notebook; no ARFF conversion or extra export snippet is needed.

| Column | Meaning |
|---|---|
| `time_s` | Original quarter-note start in seconds |
| `end_time_s` | Next quarter start; final quarter ends at the song's end marker |
| `segment`, `section` | Segment occurrence and its section label |
| `bar`, `quarter` | Original bar within this segment and quarter 1–4 |
| `chord` | **Original-pitch chord used by the HMM** |
| `bpm`, `key` | Original segment tempo and key |
| `chord_C`, `key_C` | C-normalized labels for optional inspection |
| `transpose_semitones` | Shift used only for those optional C-normalized columns |

These are quarter-level chord annotations, not individual notes or octaves. `end_time_s` defines the matching feature interval. You can stop after this step.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)

for song_id, clean in cleaned_songs.items():
    song = clean.copy()
    normalized = transposed_songs[song_id]
    song["chord_C"] = normalized.chord
    song["key_C"] = normalized.key
    song["transpose_semitones"] = normalized.transpose_semitones
    file_path = OUTPUT / f"{song_id}_ground_truth.csv"
    temporary = file_path.with_suffix(".tmp")
    song.to_csv(temporary, index=False)
    temporary.replace(file_path)

print("Cleaned CSV files written:", len(cleaned_songs))
print("HMM input folder:", OUTPUT)


Cleaned CSV files written: 1000
HMM input folder: /content/drive/MyDrive/Femman/DT2470 Music Informatics/DT2470 Project/full_data/aam_cleaned_csv


# Optional inspection and debugging
Everything below runs **after export** and is not needed to create the CSVs.

## 7a. Summarize the repairs
For the first-1,000 v1.1.0 release, expect 75 affected songs, 128 removed bars and 512 removed quarter-note rows. These totals are calculated from your files; they do not control which rows are removed.

In [ ]:
summary_rows = []
for song_id, song in cleaned_songs.items():
    summary_rows.append({
        "song_id": song_id,
        "removed_bars": len(removed_rows[song_id]) // 4,
        "clean_rows": len(song),
        "distinct_bpms": song.bpm.nunique()
    })

summary = pd.DataFrame(summary_rows)
print("Songs repaired:", (summary.removed_bars > 0).sum())
print("Extra bars removed:", summary.removed_bars.sum())
print("Quarter-note rows removed:", summary.removed_bars.sum() * 4)
print("Quarter-note rows retained:", summary.clean_rows.sum())
print("Songs with changing BPM (all kept):", (summary.distinct_bpms > 1).sum())
display(summary.loc[summary.removed_bars > 0].head(10))

Songs repaired: 75
Extra bars removed: 128
Quarter-note rows removed: 512
Quarter-note rows retained: 293800
Songs with changing BPM (all kept): 485


,song_id,removed_bars,clean_rows,distinct_bpms
1,0002,1,272,2
6,0007,3,280,1
34,0035,2,256,2
61,0062,1,480,1
62,0063,3,256,2
69,0070,1,312,2
74,0075,2,256,1
75,0076,1,144,1
89,0090,2,360,1
106,0107,3,192,2


## 7b. Inspect one song
Change `song_id` to inspect a different song. In 0002, extra bar 13 of the segment starting near 97.291210 s overlaps the next segment, starting near 118.010635 s. Sorting the original timestamps would mix these blocks, so the cleaner uses their original order.

In [ ]:
song_id = "0002"
print("Removed rows:")
display(removed_rows[song_id])

print("Cleaned rows near 118 seconds:")
song = cleaned_songs[song_id]
display(song.loc[song.time_s.between(116, 120)])

print("Transposition: one row per segment")
columns = ["segment", "original_key", "key", "transpose_semitones", "original_chord", "chord"]
display(transposed_songs[song_id].groupby("segment", sort=False).head(1)[columns])

Removed rows:


,time_s,bar,quarter,chord,segment
0,118.010630,13,1,F#maj,5
1,118.442280,13,2,F#maj,5
2,118.873940,13,3,F#maj,5
3,119.305595,13,4,F#maj,5


Cleaned rows near 118 seconds:


,time_s,end_time_s,segment,section,bar,quarter,chord,bpm,key
172,116.284010,116.715670,5,B,12,1,Bmaj,139,Bmaj
173,116.715670,117.147320,5,B,12,2,Bmaj,139,Bmaj
174,117.147320,117.578980,5,B,12,3,Bmaj,139,Bmaj
175,117.578980,118.010635,5,B,12,4,Bmaj,139,Bmaj
176,118.010635,118.442290,6,B,1,1,F#maj,139,Bmaj
177,118.442290,118.873950,6,B,1,2,F#maj,139,Bmaj
178,118.873950,119.305600,6,B,1,3,F#maj,139,Bmaj
179,119.305600,119.737250,6,B,1,4,F#maj,139,Bmaj
180,119.737250,120.168910,6,B,2,1,Emaj,139,Bmaj


Transposition: one row per segment


,segment,original_key,key,transpose_semitones,original_chord,chord
0,1,G#min,Cmin,-8,F#maj,A#maj
32,2,G#min,Cmin,-8,Emaj,G#maj
64,3,G#min,Cmin,-8,Emaj,G#maj
96,4,G#min,Cmin,-8,Emaj,G#maj
128,5,Bmaj,Cmaj,-11,F#maj,Gmaj
176,6,Bmaj,Cmaj,-11,F#maj,Gmaj
224,7,Bmaj,Cmaj,-11,F#maj,Gmaj


## 7c. Optional detailed checks
Re-cleaning must remove nothing. Transposed chords and keys must shift back to their original labels. These checks validate structure and transformation, not whether the publisher's chord labels match the audio.

In [ ]:
for song_id, song in cleaned_songs.items():
    columns = ["time_s", "bar", "quarter", "chord"]
    again, removed = remove_extra_bars(song[columns], segments[song_id])
    assert removed.empty, song_id
    pd.testing.assert_frame_equal(again, cleaned_beats[song_id], check_dtype=False)

for song_id, song in transposed_songs.items():
    for row in song.itertuples(index=False):
        assert transpose_label(row.chord, -row.transpose_semitones) == row.original_chord, song_id
        assert transpose_label(row.key, -row.transpose_semitones) == row.original_key, song_id

print("All detailed checks passed.")

All detailed checks passed.


## 8. Optional chroma rotation example
The connected HMM reads `chord` in the original pitch and uses its own **candidate chord-root rotation**. It does not use `chord_C`, `key_C` or `transpose_semitones` for training or inference. Those columns are retained for your earlier normalization experiments.

If you separately experiment with the C-normalized targets, rotate the matching chroma by `transpose_semitones` too. Changing labels alone would mismatch features and targets. Rotate each group of 12 separately for the 24-feature bass/middle representation.

This is an illustration, not an extra preprocessing step required by the connected pipeline. Applying ground-truth segment-key shifts at test time would use information unavailable in a new recording.


In [ ]:
ground_truth = pd.read_csv(OUTPUT / "0002_ground_truth.csv")
display(ground_truth.head(8))

intervals = ground_truth[["time_s", "end_time_s"]].to_numpy()
chord_labels = ground_truth["chord"].to_numpy()  # actual HMM targets
normalized_labels = ground_truth["chord_C"].to_numpy()  # optional experiment
shifts = ground_truth["transpose_semitones"].to_numpy()


def transpose_chroma(chroma_vector, shift):
    chroma_vector = np.asarray(chroma_vector)
    assert chroma_vector.ndim == 1 and len(chroma_vector) in [12, 24]
    groups = chroma_vector.reshape(-1, 12)
    return np.roll(groups, int(shift), axis=1).reshape(-1)


# Example: a vector with energy only at D becomes energy only at C.
example_chroma = np.zeros(12)
example_chroma[2] = 1
print("D shifted down by 2:", transpose_chroma(example_chroma, -2))


,time_s,end_time_s,segment,section,bar,quarter,chord,bpm,key,chord_C,key_C,transpose_semitones
0,0.000000,0.431655,1,A,1,1,F#maj,139,G#min,A#maj,Cmin,-8
1,0.431655,0.863309,1,A,1,2,F#maj,139,G#min,A#maj,Cmin,-8
2,0.863309,1.294964,1,A,1,3,F#maj,139,G#min,A#maj,Cmin,-8
3,1.294964,1.726619,1,A,1,4,F#maj,139,G#min,A#maj,Cmin,-8
4,1.726619,2.158273,1,A,2,1,G#min,139,G#min,Cmin,Cmin,-8
5,2.158273,2.589928,1,A,2,2,G#min,139,G#min,Cmin,Cmin,-8
6,2.589928,3.021583,1,A,2,3,G#min,139,G#min,Cmin,Cmin,-8
7,3.021583,3.453237,1,A,2,4,G#min,139,G#min,Cmin,Cmin,-8


D shifted down by 2: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
